# Đánh giá Hệ thống Acne Advisor AI bằng RAGAS

Notebook chạy đánh giá định lượng toàn diện hệ thống **Acne Advisor AI** theo phương pháp **RAGAS** (*Retrieval Augmented Generation Assessment*, Es et al., 2023, `arXiv:2309.15217`).

### Các chỉ số được đo:
1. **Faithfulness** (*Độ trung thực với bằng chứng*): phát hiện ảo giác (hallucination).
2. **Answer Relevancy** (*Độ liên quan câu trả lời*): đo mức độ bám sát câu hỏi người dùng.
3. **Context Precision** (*Độ chính xác ngữ cảnh*): đo độ chính xác xếp hạng top ngữ cảnh truy hồi.
4. **Context Recall** (*Độ bao phủ ngữ cảnh*): đo ngữ cảnh truy hồi có đủ ý theo đáp án chuẩn.
5. **Answer Correctness** (*Độ chính xác câu trả lời*): kết hợp độ chính xác sự kiện (75%) và độ tương đồng ngữ nghĩa (25%) so với đáp án chuẩn.
6. **Safe Abstention Accuracy** (*Tỷ lệ từ chối an toàn*): kiểm tra khả năng từ chối đúng ca ngoài phạm vi / thiếu bằng chứng.

Kết quả chi tiết được hiển thị trực tiếp ngay bên dưới các cell và đồng thời xuất ra **`evaluation/results/ragas_evaluation_report.md`**.

In [1]:
# 1. Khởi tạo đường dẫn dự án và nạp biến môi trường (.env)
from __future__ import annotations
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'evaluation':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

load_dotenv(PROJECT_ROOT / '.env')
print('Project root:', PROJECT_ROOT)
print('GOOGLE_API_KEY configured:', bool(os.getenv('GOOGLE_API_KEY')))
print('QDRANT_URL:', os.getenv('QDRANT_URL', 'http://127.0.0.1:6333'))

Project root: C:\Study\Acne Advisor AI\acne-agent-system
GOOGLE_API_KEY configured: True
QDRANT_URL: http://localhost:6333


In [2]:
# 2. Cấu hình tham số đánh giá và kiểm tra Knowledge Build hiện hành
import json
import pandas as pd
from evaluation.ragas_runner import (
    DEFAULT_DATASET_PATH,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_FALLBACK_JUDGE_MODELS,
    DEFAULT_JUDGE_MODEL,
    DEFAULT_RESULTS_DIR,
    export_evaluation_artifacts,
    load_evaluation_dataset,
    render_markdown_report,
    run_system_on_dataset,
    score_records_with_ragas,
    summarize_evaluation,
)
from src.knowledge.versioning import resolve_active_knowledge_build_id

DATASET_PATH = PROJECT_ROOT / DEFAULT_DATASET_PATH
RESULTS_DIR = PROJECT_ROOT / DEFAULT_RESULTS_DIR
SYSTEM_MODEL = 'gemini-3.5-flash-lite'  # Model trả lời của hệ thống
JUDGE_PROVIDER = 'google'
JUDGE_MODEL = DEFAULT_JUDGE_MODEL  # gemini-3.6-flash (độc lập với generator, có fallback chuỗi Gemini 3.5)
FALLBACK_JUDGE_MODELS = DEFAULT_FALLBACK_JUDGE_MODELS
EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
REQUEST_DELAY_SECONDS = 1.5
MAX_CASES = None

kb_build_id = resolve_active_knowledge_build_id()
print('Active KB Build ID:', kb_build_id)
print('System Model:', SYSTEM_MODEL, '| Judge:', f'{JUDGE_PROVIDER}/{JUDGE_MODEL}', '| Fallback Judges:', FALLBACK_JUDGE_MODELS)

C:\Study\Acne Advisor AI\acne-agent-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active KB Build ID: b768fae999391f47760a
System Model: gemini-3.5-flash-lite | Judge: google/gemini-3.6-flash | Fallback Judges: ('gemini-3.5-flash', 'gemini-3.5-flash-lite')


In [3]:
# 3. Nạp và xem tổng quan bộ câu hỏi đánh giá mới
from IPython.display import display

dataset_meta, cases = load_evaluation_dataset(DATASET_PATH)
if MAX_CASES is not None:
    cases = cases[:MAX_CASES]

df_cases = pd.DataFrame([
    {'case_id': c.case_id, 'category': c.category, 'expected': c.expected_behavior, 'question': c.user_input}
    for c in cases
])
print(f'Tổng số câu hỏi sẽ chạy: {len(cases)}')
display(df_cases)

Tổng số câu hỏi sẽ chạy: 50


,case_id,category,expected,question
0,NEW-SYM-01,symptoms_and_types,answer,Mụn nang (cyst) và mụn cục (nodule) có đặc điể...
1,NEW-SYM-02,symptoms_and_types,answer,Trứng cá mạch lươn thường gặp ở đối tượng nào ...
2,NEW-SYM-03,symptoms_and_types,answer,Sẹo lõm sau mụn trứng cá được chia thành những...
3,NEW-SYM-04,symptoms_and_types,answer,Trứng cá tối cấp (Acne fulminans) có biểu hiện...
4,NEW-SYM-05,symptoms_and_types,answer,Mụn trứng cá mức độ nhẹ và mức độ trung bình –...
5,NEW-SYM-06,symptoms_and_types,answer,Trứng cá do cào gãi có biểu hiện gì và thường ...
6,NEW-SYM-07,symptoms_and_types,answer,Mụn mủ đầu mặt ở trẻ sơ sinh (neonatal cephali...
7,NEW-SYM-08,symptoms_and_types,answer,Mụn trứng cá ở người trưởng thành (adult acne)...
8,NEW-CAU-01,causes_and_mechanisms,answer,Khí hậu nóng ẩm và thời tiết mùa hè ảnh hưởng ...
9,NEW-CAU-02,causes_and_mechanisms,answer,Chu trình chăm sóc da không phù hợp có thể tác...


In [4]:
# 4. Chạy hệ thống Acne Advisor AI trên bộ câu hỏi (tắt cache để đo thực tế)
from IPython.display import display

system_records = await run_system_on_dataset(cases, llm_provider='gemini', llm_model=SYSTEM_MODEL, verbose=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'ragas_system_records.json').write_text(
    json.dumps(system_records, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
df_sys = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'action': r['action'],
        'contexts': len(r['retrieved_contexts']),
        'sources': len(r['sources']),
        'latency_ms': r['latency_ms'],
        'error': r['error'],
    }
    for r in system_records
])
display(df_sys)


Loading weights: 100%|██████████| 393/393 [00:05<00:00, 71.38it/s]


[System 1/50] NEW-SYM-01 | action=generate | contexts=1 | 18540ms
[System 2/50] NEW-SYM-02 | action=generate | contexts=1 | 20503ms
[System 3/50] NEW-SYM-03 | action=generate | contexts=1 | 36658ms
[System 4/50] NEW-SYM-04 | action=generate | contexts=2 | 16485ms
[System 5/50] NEW-SYM-05 | action=generate | contexts=1 | 16484ms
[System 6/50] NEW-SYM-06 | action=generate | contexts=1 | 30473ms
[System 7/50] NEW-SYM-07 | action=generate | contexts=2 | 24236ms


Primary LLM (gemini/gemini-3.5-flash-lite) failed: Retry exhausted for provider gemini:gemini-3.5-flash-lite.
Agent action decision failed safely: Retry exhausted for provider gemini:gemini-3.5-flash-lite.


[System 8/50] NEW-SYM-08 gặp lỗi tạm thời từ provider (attempt 1/3), chờ 12s rồi thử lại...


Primary LLM (gemini/gemini-3.5-flash-lite) failed: Retry exhausted for provider gemini:gemini-3.5-flash-lite.
Agent action decision failed safely: Retry exhausted for provider gemini:gemini-3.5-flash-lite.


[System 8/50] NEW-SYM-08 gặp lỗi tạm thời từ provider (attempt 2/3), chờ 24s rồi thử lại...
[System 8/50] NEW-SYM-08 | action=generate | contexts=3 | 13296ms
[System 9/50] NEW-CAU-01 | action=generate | contexts=2 | 18768ms
[System 10/50] NEW-CAU-02 | action=generate | contexts=2 | 34309ms
[System 11/50] NEW-CAU-03 | action=generate | contexts=2 | 29395ms
[System 12/50] NEW-CAU-04 | action=generate | contexts=2 | 14205ms
[System 13/50] NEW-CAU-05 | action=generate | contexts=1 | 26021ms
[System 14/50] NEW-CAU-06 | action=generate | contexts=2 | 32085ms
[System 15/50] NEW-CAU-07 | action=generate | contexts=1 | 21271ms
[System 16/50] NEW-CMP-01 | action=generate | contexts=4 | 29510ms
[System 17/50] NEW-CMP-02 | action=generate | contexts=2 | 17153ms
[System 18/50] NEW-CMP-03 | action=generate | contexts=2 | 30808ms
[System 19/50] NEW-CMP-04 | action=generate | contexts=3 | 35706ms
[System 20/50] NEW-CMP-05 | action=generate | contexts=5 | 20790ms
[System 21/50] NEW-CMP-06 | action=gene

,case_id,action,contexts,sources,latency_ms,error
0,NEW-SYM-01,generate,1,1,18540.31,None
1,NEW-SYM-02,generate,1,1,20502.62,None
2,NEW-SYM-03,generate,1,1,36658.49,None
3,NEW-SYM-04,generate,2,2,16485.08,None
4,NEW-SYM-05,generate,1,2,16483.79,None
5,NEW-SYM-06,generate,1,1,30473.16,None
6,NEW-SYM-07,generate,2,1,24236.31,None
7,NEW-SYM-08,generate,3,4,13296.15,None
8,NEW-CAU-01,generate,2,1,18767.57,None
9,NEW-CAU-02,generate,2,1,34308.91,None


In [ ]:
# 5. Chấm điểm bằng RAGAS (5 chỉ số cốt lõi + Safe Abstention Accuracy)
from IPython.display import display

scored_records = await score_records_with_ragas(
    system_records,
    judge_model=JUDGE_MODEL,
    judge_provider=JUDGE_PROVIDER,
    fallback_judge_models=FALLBACK_JUDGE_MODELS,
    embedding_model=EMBEDDING_MODEL,
    request_delay_seconds=REQUEST_DELAY_SECONDS,
    verbose=True,
)
summary = summarize_evaluation(
    scored_records,
    dataset_meta=dataset_meta,
    judge_model=f'{JUDGE_PROVIDER}/{JUDGE_MODEL}',
    embedding_model=EMBEDDING_MODEL,
    kb_build_id=kb_build_id,
)
df_summary = pd.DataFrame([
    {'metric': k, **v} for k, v in summary['ragas_summary'].items()
])
display(df_summary)
df_per_case = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'category': r['category'],
        'action': r['action'],
        **(r.get('ragas_scores') or {}),
        'abstention_ok': r.get('abstention_correct'),
        'latency_ms': r['latency_ms'],
    }
    for r in scored_records
])
display(df_per_case)
print('Safe Abstention:', summary['safe_abstention'])
print('Latency (ms):', summary['latency_ms'])
print('Token Usage:', summary['token_usage'])

In [ ]:
# 6. Xuất báo cáo chi tiết ra file Markdown (.md), JSON thô và hiển thị trực tiếp tại Notebook
from IPython.display import Markdown, display

report_path, raw_path = export_evaluation_artifacts(
    summary,
    output_dir=RESULTS_DIR,
    report_filename='ragas_evaluation_report.md',
    raw_filename='ragas_evaluation_raw.json',
)
print('Đã xuất báo cáo Markdown:', report_path)
print('Đã xuất dữ liệu thô JSON:', raw_path)
report_md = render_markdown_report(summary)
preview_md = report_md.split('## 5. Chi tiết câu trả lời')[0].strip()
display(Markdown(preview_md))